# WOE/IV 계산 및 Drive 저장

Google Colab에서 실행합니다. `ml_dashboard_representatives.json`에 등록된 ml-01 이상 실험을 처리합니다. (ml-00은 테스트용 프로토타입으로 제외)

**입력 파일**
- `MyDrive/Graph_AML/ml/ml_dashboard_representatives.json` — 처리 대상 실험 목록
- `MyDrive/Graph_AML/ml/{ml_folder}/{run_id}/{prefix}_feature_columns.json` — 계산 대상 컬럼
- `MyDrive/Graph_AML/ml/{ml_folder}/{mlNN}_feature_catalog.csv` — 피처 설명 (미등록 피처 체크용)
- `MyDrive/Graph_AML/data/ml/*.parquet` — 데이터 (1개, 모든 실험 공유)

**저장 파일** (`MyDrive/Graph_AML/data/ml/woe_iv/{ml-NN}/` 하위)
- `iv_summary.json` — feature별 IV 요약
- `bin_table.json` — 전체 bin 통계 (WOE 차트용)
- `woe_meta.json` — 계산 메타데이터

**캐시** (`MyDrive/Graph_AML/data/ml/woe_iv/woe_iv_cache.json`)
- 대표 실험 prefix 변경 시에만 재계산

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## CONFIG — 필요시 이 섹션만 수정하세요

In [2]:
from pathlib import Path

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
ML_DIR         = DRIVE_BASE_DIR / "ml"
DATA_ML_DIR    = DRIVE_BASE_DIR / "data" / "ml"
WOE_IV_DIR     = DATA_ML_DIR / "woe_iv"   # 산출물 저장 루트

REPS_PATH  = ML_DIR / "ml_dashboard_representatives.json"
CACHE_PATH = WOE_IV_DIR / "woe_iv_cache.json"

# True = 캐시 무시하고 전부 재계산
FORCE_RECOMPUTE = False

# 특정 폴더만 처리 (빈 리스트 = 모든 실험)
# 예: ["ml-01", "ml-02"]
ONLY_FOLDERS: list[str] = []

# 전체 데이터 사용 여부 (False = 200k 샘플, 탐색용)
FULL_RUN = True
N_BINS   = 10
N_JOBS   = -1

## Engine — 수정하지 마세요

In [3]:
from __future__ import annotations

import datetime
import json
import re
import time
from dataclasses import dataclass
from typing import List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import pyarrow.parquet as _pq
from joblib import Parallel, delayed

# ── 상수 ──────────────────────────────────────────────────────────────────

OTHER_LABEL   = "__OTHER__"
MISSING_LABEL = "__MISSING__"

BIN_COLS = [
    "feature_name", "feature_type", "bin_id", "bin_label",
    "count", "positive_count", "negative_count", "positive_rate",
    "total_positive_rate", "positive_dist", "negative_dist", "woe", "iv_bin",
    "missing_flag", "binning_method",
]

IV_COLS = [
    "feature_name", "feature_type", "binning_method", "n_bins",
    "total_count", "positive_count", "negative_count", "iv", "iv_strength",
    "missing_count", "unique_count", "note",
]

_RESULT_FILES = ["iv_summary.json", "bin_table.json", "woe_meta.json"]


@dataclass
class Config:
    sample_mode: bool = True
    sample_n: int = 200_000
    random_state: int = 42
    n_bins_numeric: int = 10
    cat_top_n: int = 50
    cat_min_count: int = 100
    smoothing_alpha: float = 0.5
    n_jobs: int = -1


# ── Binning helpers ────────────────────────────────────────────────────────

def fit_numeric_edges(s: pd.Series, n_bins: int) -> Optional[np.ndarray]:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty or s.nunique() < 2:
        return None
    q = min(n_bins, int(s.nunique()))
    try:
        _, edges = pd.qcut(s, q=q, retbins=True, duplicates="drop")
        edges = np.array(edges, dtype=float)
    except Exception:
        uniq = np.sort(s.unique())
        if len(uniq) < 2:
            return None
        edges = np.concatenate([uniq[:1], (uniq[:-1] + uniq[1:]) / 2, uniq[-1:]])
    if len(edges) < 3:
        return None
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def apply_numeric_edges(s: pd.Series, edges: np.ndarray) -> pd.Series:
    return pd.cut(pd.to_numeric(s, errors="coerce"), bins=edges, include_lowest=True, duplicates="drop")


def fit_cat_levels(s: pd.Series, top_n: int, min_count: int) -> Set[object]:
    counts = s.astype("object").value_counts(dropna=True)
    if counts.empty:
        return set()
    keep = counts[counts >= min_count]
    return set((keep.head(top_n) if not keep.empty else counts.head(top_n)).index)


def apply_cat_levels(s: pd.Series, keep_set: Set[object]) -> pd.Series:
    so = s.astype("object")
    return so.where(so.isin(keep_set) | so.isna(), other=OTHER_LABEL)


def compute_bin_stats(bin_series: pd.Series, y: pd.Series, alpha: float,
                      total_pos: float = None, total_neg: float = None) -> pd.DataFrame:
    labels = bin_series.astype("object").where(~bin_series.isna(), MISSING_LABEL).map(str)
    g = (
        pd.DataFrame({"bin_label": labels.values, "label": y.values})
        .groupby("bin_label", sort=False)["label"]
        .agg(count="size", positive_count="sum")
        .reset_index()
    )
    g["negative_count"] = g["count"] - g["positive_count"]
    if total_pos is None:
        total_pos = float(g["positive_count"].sum())
    if total_neg is None:
        total_neg = float(g["negative_count"].sum())
    n_bins    = max(len(g), 1)
    g["positive_rate"]       = g["positive_count"] / g["count"].replace(0, np.nan)
    g["total_positive_rate"] = total_pos / max(float(g["count"].sum()), 1.0)
    pos_dist = (g["positive_count"] + alpha) / (total_pos + alpha * n_bins)
    neg_dist = (g["negative_count"] + alpha) / (total_neg + alpha * n_bins)
    g["positive_dist"] = pos_dist
    g["negative_dist"] = neg_dist
    g["woe"]    = np.log(pos_dist / neg_dist)
    g["iv_bin"] = (pos_dist - neg_dist) * g["woe"]
    g["missing_flag"] = g["bin_label"].eq(MISSING_LABEL)
    return g


def iv_strength(iv: float) -> str:
    if pd.isna(iv): return "na"
    if iv < 0.02:   return "useless"
    if iv < 0.10:   return "weak"
    if iv < 0.30:   return "medium"
    if iv < 0.50:   return "strong"
    return "suspicious"


def _process_feature(df: pd.DataFrame, feature_name: str, feature_type: str, cfg: Config,
                     total_pos: float = None, total_neg: float = None):
    y             = df["label"]
    s             = df[feature_name]
    unique_count  = int(s.nunique(dropna=True))
    missing_count = int(s.isna().sum())

    if feature_type == "numeric":
        edges = fit_numeric_edges(s, cfg.n_bins_numeric)
        if edges is None:
            return None, {
                "feature_name": feature_name, "feature_type": feature_type,
                "binning_method": "quantile", "n_bins": 0,
                "total_count": len(df), "positive_count": int(y.sum()),
                "negative_count": int((y == 0).sum()),
                "iv": np.nan, "iv_strength": "na",
                "missing_count": missing_count, "unique_count": unique_count,
                "note": "skipped: not enough unique numeric values",
            }
        binned     = apply_numeric_edges(s, edges)
        bin_id_map = {str(cat): i for i, cat in enumerate(binned.cat.categories)}
        bin_id_map[MISSING_LABEL] = -1
        method = f"quantile_n={len(edges) - 1}"
    else:
        keep_set   = fit_cat_levels(s, cfg.cat_top_n, cfg.cat_min_count)
        binned     = apply_cat_levels(s, keep_set)
        ordered    = sorted([str(x) for x in keep_set]) + [OTHER_LABEL, MISSING_LABEL]
        bin_id_map = {label: i for i, label in enumerate(ordered)}
        method = f"cat_top_n={cfg.cat_top_n}_min_count={cfg.cat_min_count}"

    tbl = compute_bin_stats(binned, y, cfg.smoothing_alpha, total_pos=total_pos, total_neg=total_neg)
    tbl["feature_name"]   = feature_name
    tbl["feature_type"]   = feature_type
    tbl["binning_method"] = method
    tbl["bin_id"] = tbl["bin_label"].map(lambda x: bin_id_map.get(str(x), -99))
    tbl = tbl[BIN_COLS]

    iv_total = float(tbl["iv_bin"].sum())
    notes = []
    if int((tbl["positive_count"] == 0).sum()):
        notes.append(f"zero_pos_bins={int((tbl['positive_count']==0).sum())}")
    if int((tbl["negative_count"] == 0).sum()):
        notes.append(f"zero_neg_bins={int((tbl['negative_count']==0).sum())}")
    if feature_type == "categorical" and (tbl["bin_label"] == OTHER_LABEL).any():
        notes.append(f"OTHER_count={int(tbl.loc[tbl['bin_label'].eq(OTHER_LABEL), 'count'].sum())}")

    iv_row = {
        "feature_name":   feature_name,
        "feature_type":   feature_type,
        "binning_method": method,
        "n_bins":         int(len(tbl)),
        "total_count":    int(tbl["count"].sum()),
        "positive_count": int(tbl["positive_count"].sum()),
        "negative_count": int(tbl["negative_count"].sum()),
        "iv":             iv_total,
        "iv_strength":    iv_strength(iv_total),
        "missing_count":  int(tbl.loc[tbl["missing_flag"], "count"].sum()),
        "unique_count":   unique_count,
        "note":           "; ".join(notes),
    }
    return tbl, iv_row


def classify_explicit_features(
    df: pd.DataFrame, feature_columns: list[str]
) -> Tuple[List[str], List[str]]:
    numeric, categorical = [], []
    for col in feature_columns:
        if col not in df.columns:
            print(f"  WARNING: '{col}' parquet에 없음, 건너뜀")
            continue
        if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 5:
            numeric.append(col)
        else:
            categorical.append(col)
    return numeric, categorical


def compute_woe_iv_explicit(
    df: pd.DataFrame, feature_columns: list[str], cfg: Config
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    numeric_features, categorical_features = classify_explicit_features(df, feature_columns)
    all_features = (
        [(f, "numeric")     for f in numeric_features] +
        [(f, "categorical") for f in categorical_features]
    )
    y         = df["label"]
    total_pos = float(y.sum())
    total_neg = float(len(y) - total_pos)
    results = Parallel(n_jobs=cfg.n_jobs, prefer="threads")(
        delayed(_process_feature)(df, feat, ftype, cfg, total_pos, total_neg)
        for feat, ftype in all_features
    )
    bin_parts = [tbl for tbl, _ in results if tbl is not None]
    iv_rows   = [row for _, row in results]
    bin_df = pd.concat(bin_parts, ignore_index=True) if bin_parts else pd.DataFrame(columns=BIN_COLS)
    iv_df  = (
        pd.DataFrame(iv_rows)[IV_COLS]
        .sort_values("iv", ascending=False)
        .reset_index(drop=True)
    )
    return bin_df, iv_df


# ── 캐시 ──────────────────────────────────────────────────────────────────

def load_cache(cache_path) -> dict:
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}


def save_cache(cache: dict, cache_path) -> None:
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")


def needs_recompute(cache: dict, woe_folder: str, prefix: str, save_dir) -> bool:
    entry = cache.get(woe_folder)
    if entry is None or entry.get("prefix") != prefix:
        return True
    missing = [f for f in _RESULT_FILES if not (save_dir / f).exists()]
    if missing:
        print(f"  ⚠ 캐시는 유효하나 결과 파일 누락: {missing} → 재계산")
        return True
    return False


# ── 경로 헬퍼 ─────────────────────────────────────────────────────────────

def _woe_iv_folder_name(ml_folder: str) -> str:
    m = re.match(r"(ml-\d+)", ml_folder)
    return m.group(1) if m else ml_folder


def _catalog_source_path(ml_folder: str, ml_dir) -> "Path":
    label = _woe_iv_folder_name(ml_folder).replace("-", "")
    return ml_dir / ml_folder / f"{label}_feature_catalog.csv"


def _is_valid_rep(rep: dict) -> bool:
    woe_folder = _woe_iv_folder_name(rep.get("ml_folder", ""))
    return bool(re.match(r"ml-\d+", woe_folder)) and woe_folder != "ml-00"


def _artifact_prefix(rep: dict) -> str:
    return f"{rep['experiment_id']}__{rep['run_id']}__{rep['model_run_id']}"


# ── 데이터 로더 ───────────────────────────────────────────────────────────

def find_ml_parquet(data_ml_dir) -> Optional["Path"]:
    parquets = sorted(data_ml_dir.glob("*.parquet"))
    if not parquets:
        return None
    if len(parquets) > 1:
        print(f"  WARNING: {len(parquets)}개 parquet 발견 → 첫 번째 사용: {parquets[0].name}")
    return parquets[0]


def load_feature_columns(ml_dir, ml_folder: str, prefix: str, run_id: str) -> list[str]:
    path = ml_dir / ml_folder / run_id / f"{prefix}_feature_columns.json"
    if not path.exists():
        raise FileNotFoundError(f"feature_columns.json 없음: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    return data.get("feature_columns", data) if isinstance(data, dict) else list(data)


print("Engine 로드 완료")

Engine 로드 완료


## 실행

In [ ]:
# ── 초기화 ────────────────────────────────────────────────────────────────
WOE_IV_DIR.mkdir(parents=True, exist_ok=True)

if not REPS_PATH.exists():
    raise FileNotFoundError(f"representatives.json 없음: {REPS_PATH}")

reps       = json.loads(REPS_PATH.read_text(encoding="utf-8"))
valid_reps = [r for r in reps if _is_valid_rep(r)]

if ONLY_FOLDERS:
    valid_reps = [r for r in valid_reps
                  if _woe_iv_folder_name(r["ml_folder"]) in ONLY_FOLDERS]

print(f"처리 대상 실험: {[_woe_iv_folder_name(r['ml_folder']) for r in valid_reps]}")

cache = load_cache(CACHE_PATH)

cfg = Config(
    sample_mode=not FULL_RUN,
    n_bins_numeric=N_BINS,
    n_jobs=N_JOBS,
)


# ── 실험별 처리 루프 ──────────────────────────────────────────────────────
for rep in valid_reps:
    ml_folder      = rep["ml_folder"]
    run_id         = rep["run_id"]
    prefix         = _artifact_prefix(rep)                    # experiment_id__run_id__model_run_id
    woe_folder     = _woe_iv_folder_name(ml_folder)           # e.g. "ml-01"
    save_dir       = WOE_IV_DIR / woe_folder                  # data/ml/woe_iv/ml-01/
    catalog_source = _catalog_source_path(ml_folder, ML_DIR)  # ml/{ml_folder}/{mlNN}_feature_catalog.csv

    save_dir.mkdir(parents=True, exist_ok=True)

    # 카탈로그 로드 (소스에서 직접 읽기)
    if catalog_source.exists():
        catalog_df = pd.read_csv(catalog_source, encoding="utf-8-sig")
        catalog_df.columns = catalog_df.columns.str.strip()
        if "피쳐명" in catalog_df.columns:
            catalog_df = catalog_df.rename(columns={"피쳐명": "피처명"})
        registered = set(catalog_df["피처명"].tolist()) if "피처명" in catalog_df.columns else set()
    else:
        catalog_df = None
        registered = set()

    # 캐시 확인 — prefix 변경 또는 결과 파일 누락 시 재계산
    if not FORCE_RECOMPUTE and not needs_recompute(cache, woe_folder, prefix, save_dir):
        print(f"\n[{woe_folder}] ✓ 캐시 유효 — 건너뜀 (prefix={prefix})")
        continue

    print(f"\n{'='*60}")
    print(f"[{woe_folder}] WOE/IV 계산 시작  (prefix={prefix})")
    t0 = time.time()

    # feature_columns 로드
    try:
        feature_cols = load_feature_columns(ML_DIR, ml_folder, prefix, run_id)
    except FileNotFoundError as e:
        print(f"  ERROR: {e} → 건너뜀")
        continue
    print(f"  feature_columns: {len(feature_cols)}개")

    # 카탈로그 미등록 피처 출력
    if catalog_df is not None:
        unregistered = [f for f in feature_cols if f not in registered]
        if unregistered:
            n = len(unregistered)
            print(f"  ⚠ 카탈로그 미등록 {n}개: {unregistered[:5]}{'...' if n > 5 else ''}")
        else:
            print(f"  카탈로그: {len(catalog_df)}행 ({catalog_source.name})")
    else:
        unregistered = []
        print(f"  WARNING: 카탈로그 없음 ({catalog_source})")

    # parquet 로드 (필요한 컬럼만 선택하여 메모리·I/O 절감)
    parquet_path = find_ml_parquet(DATA_ML_DIR)
    if parquet_path is None:
        print(f"  ERROR: parquet 없음: {DATA_ML_DIR} → 건너뜀")
        continue
    print(f"  parquet 로드: {parquet_path.name}")
    _pq_cols   = set(_pq.read_schema(parquet_path).names)
    _load_cols = [c for c in feature_cols + ["label"] if c in _pq_cols]
    df_full    = pd.read_parquet(parquet_path, columns=_load_cols)
    print(f"  shape: {df_full.shape}  |  positive rate: {df_full['label'].mean():.5f}")
    if not FULL_RUN:
        df_full = df_full.sample(cfg.sample_n, random_state=cfg.random_state).reset_index(drop=True)
        print(f"  (샘플링 후: {len(df_full):,}행)")

    # WOE/IV 계산
    print("  WOE/IV 계산 중...")
    avail_cols = [c for c in feature_cols if c in df_full.columns]
    df_run = df_full[avail_cols + ["label"]]
    bin_df, iv_df = compute_woe_iv_explicit(df_run, feature_cols, cfg)

    elapsed = time.time() - t0
    print(f"  완료 ({elapsed:.1f}초)  |  분석 feature: {iv_df['iv'].notna().sum()}개  |  bin 행: {len(bin_df):,}")

    # 결과 저장
    woe_meta = {
        "computed_at":         datetime.datetime.now().isoformat(),
        "woe_folder":          woe_folder,
        "ml_folder":           ml_folder,
        "experiment_id":       rep["experiment_id"],
        "run_id":              run_id,
        "prefix":              prefix,
        "parquet_file":        parquet_path.name,
        "full_run":            FULL_RUN,
        "n_bins_numeric":      N_BINS,
        "n_rows":              len(df_run),
        "positive_rate":       float(df_run["label"].mean()),
        "n_features_analyzed": int(iv_df["iv"].notna().sum()),
        "unregistered_count":  len(unregistered),
        "elapsed_seconds":     round(elapsed, 1),
    }

    iv_df.to_json(save_dir / "iv_summary.json",  orient="records", force_ascii=False, indent=2)
    bin_df.to_json(save_dir / "bin_table.json",   orient="records", force_ascii=False)
    with open(save_dir / "woe_meta.json", "w", encoding="utf-8") as f:
        json.dump(woe_meta, f, indent=2, ensure_ascii=False)

    # 캐시 업데이트
    cache[woe_folder] = {
        "prefix":      prefix,
        "computed_at": woe_meta["computed_at"],
    }
    save_cache(cache, CACHE_PATH)

    print(f"  저장 위치: {save_dir}")
    for fname in _RESULT_FILES:
        fpath = save_dir / fname
        if fpath.exists():
            print(f"  {fname:<35} {fpath.stat().st_size / 1024:>7.1f} KB")

print("\n모든 실험 처리 완료")